# Estudo sobre a Paralelização do Algoritmo A*
### Análise comparativa: Serial vs Bidirecional A* vs HDA* vs PBNF
**Universidade Presbiteriana Mackenzie — FCI**


---

Este notebook reproduz todos os gráficos e tabelas do artigo, lendo diretamente os CSVs de experimento.

> **Como usar:** execute todas as células em ordem (`Run All`). Ajuste o caminho dos CSVs na célula de configuração se necessário.

## Dependências e Configuração

In [ ]:
# Instala dependências caso necessário
import subprocess, sys
for pkg in ['pandas', 'matplotlib', 'numpy', 'seaborn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# CAMINHOS DOS SEUS ARQUIVOS CSV
CSV_SERIAL        = 'serial.csv'
CSV_BIDIRECIONAL  = 'parallel_bidirectional.csv'
CSV_HDA           = 'parallel_hda.csv'
CSV_PBNF          = 'parallel_pbnf.csv'

# Número de threads por abordagem (conforme artigo)
THREADS = {'Serial': 1, 'Bidirecional A*': 2, 'HDA*': 4, 'PBNF': 4}

COLORS = {
    'Serial':         '#2c3e50',
    'Bidirecional A*':'#27ae60',
    'HDA*':           '#e67e22',
    'PBNF':           '#e74c3c',
}
MARKERS = {'Serial': 'o', 'Bidirecional A*': 's', 'HDA*': '^', 'PBNF': 'D'}
LINESTYLES = {'Serial': '--', 'Bidirecional A*': '-', 'HDA*': '-', 'PBNF': '-'}

OBSTACLE_LABELS = ['0%', '10%', '20%', '30%']
OBSTACLE_VALUES = [0, 10, 20, 30]
GRID_SIZES      = ['5000x5000', '6000x6000', '7000x7000']
GRID_LABELS     = ['5K×5K', '6K×6K', '7K×7K']

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'legend.framealpha': 0.9,
})

print('✅ Configuração carregada.')

✅ Configuração carregada.


## 1. Carregamento e Preparação dos Dados

In [ ]:
def load(path, label):
    df = pd.read_csv(path, sep=';')
    df.columns = df.columns.str.strip()
    df['abordagem']   = label
    df['grid_size']   = df['Tamanho do Grid'].str.extract(r'(\d+x\d+)')[0]
    df['obst_pct']    = df['Percentual Obstaculos (%)'].astype(int)
    df['tempo_ms']    = pd.to_numeric(df['Tempo de Execucao (ms)'], errors='coerce')
    df['nos_exp']     = pd.to_numeric(df['Nos Expandidos'], errors='coerce')
    df['custo_total'] = pd.to_numeric(df['Custo Total do Caminho'], errors='coerce')
    return df

raw = pd.concat([
    load(CSV_SERIAL,       'Serial'),
    load(CSV_BIDIRECIONAL, 'Bidirecional A*'),
    load(CSV_HDA,          'HDA*'),
    load(CSV_PBNF,         'PBNF'),
], ignore_index=True)

# Estatísticas agregadas (média + desvio padrão) por configuração
agg = raw.groupby(['abordagem', 'grid_size', 'obst_pct']).agg(
    tempo_medio = ('tempo_ms', 'mean'),
    tempo_std   = ('tempo_ms', 'std'),
    nos_medio   = ('nos_exp', 'mean'),
    nos_std     = ('nos_exp', 'std'),
    custo_medio = ('custo_total', 'mean'),
    custo_std   = ('custo_total', 'std'),
).reset_index()

# Speedup = T_serial / T_paralela (mesmo grid + obstáculo)
serial_ref = agg[agg['abordagem'] == 'Serial'][['grid_size', 'obst_pct', 'tempo_medio']]\
               .rename(columns={'tempo_medio': 'tempo_serial'})
agg = agg.merge(serial_ref, on=['grid_size', 'obst_pct'], how='left')
agg['speedup']    = agg['tempo_serial'] / agg['tempo_medio']
agg['eficiencia'] = agg['speedup'] / agg['abordagem'].map(THREADS)

print(f'✅ {len(raw)} registros carregados.')
print(f'   Abordagens: {raw["abordagem"].unique().tolist()}')
print(f'   Grids: {raw["grid_size"].unique().tolist()}')
print(f'   Obstáculos: {sorted(raw["obst_pct"].unique().tolist())}%')

FileNotFoundError: [Errno 2] No such file or directory: 'serial.csv'

## 2. Tabela de Resumo — Tempo Médio de Execução (ms) — Grid 5K×5K
> **Tabela 3 do artigo**

In [ ]:
def fmt_mean_std(row, col_mean, col_std):
    return f"{row[col_mean]:,.1f} ± {row[col_std]:,.1f}"

grid5k = agg[agg['grid_size'] == '5000x5000'].copy()

rows = []
for obst in OBSTACLE_VALUES:
    row = {'Obstáculos': f'{obst}%'}
    for ab in ['Serial', 'Bidirecional A*', 'HDA*', 'PBNF']:
        sub = grid5k[(grid5k['abordagem'] == ab) & (grid5k['obst_pct'] == obst)]
        row[ab] = fmt_mean_std(sub.iloc[0], 'tempo_medio', 'tempo_std')
    rows.append(row)

tabela3 = pd.DataFrame(rows).set_index('Obstáculos')
tabela3.style\
    .set_caption('Tabela 3 — Tempo médio de execução (ms) por abordagem — Grid 5K×5K')\
    .set_table_styles([{'selector': 'caption', 'props': [('font-weight','bold'),('font-size','13px')]}])

## 3. Figura 1 — Tempo de Execução: Serial vs Paralelos — Grid 5K×5K
> Escala logarítmica para representar as ordens de grandeza

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

grid5k = agg[agg['grid_size'] == '5000x5000']

for ab in ['Serial', 'Bidirecional A*', 'HDA*', 'PBNF']:
    sub = grid5k[grid5k['abordagem'] == ab].sort_values('obst_pct')
    ax.errorbar(
        OBSTACLE_LABELS,
        sub['tempo_medio'],
        yerr=sub['tempo_std'],
        label=ab,
        color=COLORS[ab],
        marker=MARKERS[ab],
        linestyle=LINESTYLES[ab],
        linewidth=2,
        markersize=7,
        capsize=4,
    )

ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlabel('Percentual de Obstáculos')
ax.set_ylabel('Tempo Médio de Execução (ms) — escala log')
ax.set_title('Fig. 1 — Comparação de Tempo de Execução\nSerial vs Paralelos — Grid 5K×5K')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig('fig1_tempo_5k.png', bbox_inches='tight')
plt.show()

## 4. Tabela de Speedup — Grid 5K×5K
> **Tabela 4 do artigo**

In [ ]:
rows = []
for obst in OBSTACLE_VALUES:
    row = {'Obstáculos': f'{obst}%'}
    for ab in ['Bidirecional A*', 'HDA*', 'PBNF']:
        sub = grid5k[(grid5k['abordagem'] == ab) & (grid5k['obst_pct'] == obst)]
        row[f'Speedup {ab}'] = f"{sub.iloc[0]['speedup']:.2f}×"
    rows.append(row)

tabela4 = pd.DataFrame(rows).set_index('Obstáculos')
tabela4.style\
    .set_caption('Tabela 4 — Speedup das abordagens paralelas em relação ao Serial — Grid 5K×5K')\
    .set_table_styles([{'selector': 'caption', 'props': [('font-weight','bold'),('font-size','13px')]}])

## 5. Figura 2 — Speedup das Abordagens Paralelas — Grid 5K×5K

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

grid5k = agg[agg['grid_size'] == '5000x5000']

for ab in ['Bidirecional A*', 'HDA*', 'PBNF']:
    sub = grid5k[grid5k['abordagem'] == ab].sort_values('obst_pct')
    ax.plot(
        OBSTACLE_LABELS,
        sub['speedup'],
        label=ab,
        color=COLORS[ab],
        marker=MARKERS[ab],
        linestyle='-',
        linewidth=2.5,
        markersize=8,
    )
    # Anotação do valor no ponto de pico do Bidirecional
    if ab == 'Bidirecional A*':
        for _, r in sub.iterrows():
            ax.annotate(
                f"{r['speedup']:.2f}×",
                xy=(OBSTACLE_LABELS[OBSTACLE_VALUES.index(r['obst_pct'])], r['speedup']),
                xytext=(5, 6), textcoords='offset points',
                fontsize=9, color=COLORS[ab], fontweight='bold'
            )

# Linha de referência speedup = 1
ax.axhline(1, linestyle='--', color='gray', linewidth=1.5, label='Referência (Serial = 1×)')

ax.set_xlabel('Percentual de Obstáculos')
ax.set_ylabel('Speedup (T_serial / T_paralelo)')
ax.set_title('Fig. 2 — Speedup das Abordagens Paralelas\nem relação ao Serial — Grid 5K×5K')
ax.legend()
plt.tight_layout()
plt.savefig('fig2_speedup_5k.png', bbox_inches='tight')
plt.show()

## 6. Figura 3 — Eficiência das Abordagens Paralelas — Grid 5K×5K
> E = Speedup / Nº de threads

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

grid5k = agg[agg['grid_size'] == '5000x5000']

for ab in ['Bidirecional A*', 'HDA*', 'PBNF']:
    sub = grid5k[grid5k['abordagem'] == ab].sort_values('obst_pct')
    lbl = f'{ab} ({THREADS[ab]} threads)'
    ax.plot(
        OBSTACLE_LABELS, sub['eficiencia'],
        label=lbl, color=COLORS[ab],
        marker=MARKERS[ab], linestyle='-',
        linewidth=2.5, markersize=8
    )

# Linha de eficiência ideal (100%)
ax.axhline(1, linestyle='--', color='gray', linewidth=1.5, label='Eficiência Ideal (100%)')

ax.set_xlabel('Percentual de Obstáculos')
ax.set_ylabel('Eficiência (Speedup / Nº de Threads)')
ax.set_title('Fig. 3 — Eficiência das Abordagens Paralelas — Grid 5K×5K')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_eficiencia_5k.png', bbox_inches='tight')
plt.show()

## 7. Figuras 4, 5, 6 — Nós Expandidos por Abordagem (todos os grids)
> Escala logarítmica — barras agrupadas por percentual de obstáculos

In [ ]:
ABORDAGENS   = ['Serial', 'Bidirecional A*', 'HDA*', 'PBNF']
N_AB         = len(ABORDAGENS)
BAR_WIDTH    = 0.18
X            = np.arange(len(OBSTACLE_VALUES))

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)
fig.suptitle('Figuras 4–6 — Média de Nós Expandidos por Abordagem (escala log)', fontsize=13, fontweight='bold')

for ax, (grid, glabel) in zip(axes, zip(GRID_SIZES, GRID_LABELS)):
    sub_grid = agg[agg['grid_size'] == grid]
    for i, ab in enumerate(ABORDAGENS):
        sub = sub_grid[sub_grid['abordagem'] == ab].sort_values('obst_pct')
        offset = (i - N_AB / 2 + 0.5) * BAR_WIDTH
        bars = ax.bar(
            X + offset, sub['nos_medio'],
            width=BAR_WIDTH, label=ab,
            color=COLORS[ab], alpha=0.88, edgecolor='white'
        )

    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_xticks(X)
    ax.set_xticklabels(OBSTACLE_LABELS)
    ax.set_xlabel('Percentual de Obstáculos')
    ax.set_ylabel('Nº Médio de Nós Expandidos (log)' if ax == axes[0] else '')
    ax.set_title(f'Grid {glabel}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig4_5_6_nos_expandidos.png', bbox_inches='tight')
plt.show()

## 8. Figuras 7, 8, 9 — Custo Total do Caminho (todos os grids)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)
fig.suptitle('Figuras 7–9 — Custo Total Médio do Caminho por Abordagem', fontsize=13, fontweight='bold')

for ax, (grid, glabel) in zip(axes, zip(GRID_SIZES, GRID_LABELS)):
    sub_grid = agg[agg['grid_size'] == grid]
    for i, ab in enumerate(ABORDAGENS):
        sub = sub_grid[sub_grid['abordagem'] == ab].sort_values('obst_pct')
        offset = (i - N_AB / 2 + 0.5) * BAR_WIDTH
        ax.bar(
            X + offset, sub['custo_medio'],
            width=BAR_WIDTH, label=ab,
            color=COLORS[ab], alpha=0.88, edgecolor='white'
        )

    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_xticks(X)
    ax.set_xticklabels(OBSTACLE_LABELS)
    ax.set_xlabel('Percentual de Obstáculos')
    ax.set_ylabel('Custo Total Médio do Caminho' if ax == axes[0] else '')
    ax.set_title(f'Grid {glabel}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig7_8_9_custo_total.png', bbox_inches='tight')
plt.show()

## 9. Figura 10 — Comparação Direta Entre Abordagens Paralelas — Grid 5K×5K

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

grid5k = agg[agg['grid_size'] == '5000x5000']

for ab in ['Bidirecional A*', 'HDA*', 'PBNF']:
    sub = grid5k[grid5k['abordagem'] == ab].sort_values('obst_pct')
    ax.bar(
        X + (['Bidirecional A*', 'HDA*', 'PBNF'].index(ab) - 1) * BAR_WIDTH,
        sub['tempo_medio'],
        width=BAR_WIDTH, label=ab,
        color=COLORS[ab], alpha=0.9, edgecolor='white'
    )

ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xticks(X)
ax.set_xticklabels(OBSTACLE_LABELS)
ax.set_xlabel('Percentual de Obstáculos')
ax.set_ylabel('Tempo Médio de Execução (ms) — escala log')
ax.set_title('Fig. 10 — Comparação de Tempo entre\nAbordagens Paralelas — Grid 5K×5K')
ax.legend()
plt.tight_layout()
plt.savefig('fig10_paralelas_5k.png', bbox_inches='tight')
plt.show()

## 10. Figura 11 — Bidirecional A* vs Serial por Tamanho de Grid
> Linhas contínuas = Bidirecional; linhas tracejadas = Serial

In [ ]:
GRID_COLORS = {'5000x5000': '#2980b9', '6000x6000': '#8e44ad', '7000x7000': '#16a085'}

fig, ax = plt.subplots(figsize=(9, 5))

for grid, glabel in zip(GRID_SIZES, GRID_LABELS):
    for ab, ls in [('Bidirecional A*', '-'), ('Serial', '--')]:
        sub = agg[(agg['grid_size'] == grid) & (agg['abordagem'] == ab)].sort_values('obst_pct')
        lbl = f'Bidirecional {glabel}' if ab == 'Bidirecional A*' else f'Serial {glabel}'
        ax.plot(
            OBSTACLE_LABELS, sub['tempo_medio'],
            linestyle=ls, color=GRID_COLORS[grid],
            marker=('s' if ab == 'Bidirecional A*' else 'o'),
            linewidth=2.2, markersize=7, label=lbl,
            alpha=(1.0 if ab == 'Bidirecional A*' else 0.65)
        )

ax.set_xlabel('Percentual de Obstáculos')
ax.set_ylabel('Tempo Médio de Execução (ms)')
ax.set_title('Fig. 11 — Tempo de Execução do Bidirecional A* vs Serial\npor Tamanho de Grid')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Legenda customizada
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0],[0], color=GRID_COLORS[g], linewidth=2.5, label=f'{gl}')
    for g, gl in zip(GRID_SIZES, GRID_LABELS)
] + [
    Line2D([0],[0], color='gray', linestyle='-',  linewidth=2, label='Bidirecional (sólido)'),
    Line2D([0],[0], color='gray', linestyle='--', linewidth=2, label='Serial (tracejado)'),
]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.savefig('fig11_bidir_vs_serial_todos_grids.png', bbox_inches='tight')
plt.show()

## 11. Figura 12 — Speedup do Bidirecional A* por Tamanho de Grid

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for grid, glabel in zip(GRID_SIZES, GRID_LABELS):
    sub = agg[(agg['grid_size'] == grid) & (agg['abordagem'] == 'Bidirecional A*')].sort_values('obst_pct')
    ax.plot(
        OBSTACLE_LABELS, sub['speedup'],
        color=GRID_COLORS[grid], marker='s',
        linewidth=2.5, markersize=8, label=glabel
    )
    for _, r in sub.iterrows():
        ax.annotate(
            f"{r['speedup']:.2f}×",
            xy=(OBSTACLE_LABELS[OBSTACLE_VALUES.index(r['obst_pct'])], r['speedup']),
            xytext=(4, 6), textcoords='offset points', fontsize=8,
            color=GRID_COLORS[grid]
        )

ax.axhline(1, linestyle='--', color='gray', linewidth=1.5, label='Linha base (speedup = 1)')
ax.set_xlabel('Percentual de Obstáculos')
ax.set_ylabel('Speedup (T_serial / T_bidirecional)')
ax.set_title('Fig. 12 — Speedup do Bidirecional A* por Tamanho de Grid')
ax.legend()
plt.tight_layout()
plt.savefig('fig12_speedup_bidir_por_grid.png', bbox_inches='tight')
plt.show()

## 12. Tabela 5 — Tempo Médio e Speedup — Todos os Grids e Densidades
> Tabela completa de escalabilidade

In [ ]:
rows = []
for grid, glabel in zip(GRID_SIZES, GRID_LABELS):
    for obst in OBSTACLE_VALUES:
        row = {'Grid': glabel, 'Obstáculos': f'{obst}%'}
        sub = agg[(agg['grid_size'] == grid) & (agg['obst_pct'] == obst)]
        # Serial
        s = sub[sub['abordagem'] == 'Serial'].iloc[0]
        row['Serial (ms)'] = f"{s['tempo_medio']:,.1f}"
        for ab in ['Bidirecional A*', 'HDA*', 'PBNF']:
            p = sub[sub['abordagem'] == ab].iloc[0]
            row[f'{ab} (ms)'] = f"{p['tempo_medio']:,.1f}"
            row[f'Speedup {ab}'] = f"{p['speedup']:.2f}×"
        rows.append(row)

tabela5 = pd.DataFrame(rows).set_index(['Grid', 'Obstáculos'])
tabela5.style\
    .set_caption('Tabela 5 — Tempo médio (ms) e speedup das abordagens por grid e densidade')\
    .set_table_styles([{'selector': 'caption', 'props': [('font-weight','bold'),('font-size','13px')]}])

## 13. Tabela 6 — Síntese Comparativa das Abordagens

In [ ]:
tabela6_data = {
    'Critério': [
        'Desempenho sem obstáculos',
        'Desempenho com obstáculos',
        'Escalabilidade',
        'Garantia de Otimalidade',
        'Overhead paralelo',
        'Recomendação geral',
    ],
    'Serial': [
        'Referência', 'Referência',
        'Moderada', 'Sim', 'N/A',
        'Grids sem obstáculos',
    ],
    'Bidirecional A*': [
        'Inferior (0,85×)',
        'Superior (até 5,61×)',
        'Boa',
        'Sim (MM)',
        'Baixo',
        'Grids grandes com obstáculos',
    ],
    'HDA*': [
        'Inferior (0,69×)',
        'Inferior (0,19–0,32×)',
        'Não escalável (degradação severa)',
        'Sim',
        'Alto (hashing)',
        'Não recomendado',
    ],
    'PBNF': [
        'Superior (1,79×)',
        'Inferior (0,17–0,24×)',
        'Não escalável (degradação severa)',
        'Sim',
        'Alto (nblocks)',
        'Somente sem obstáculos',
    ],
}
tabela6 = pd.DataFrame(tabela6_data).set_index('Critério')
tabela6.style\
    .set_caption('Tabela 6 — Síntese comparativa das abordagens avaliadas')\
    .set_table_styles([{'selector': 'caption', 'props': [('font-weight','bold'),('font-size','13px')]}])

## 14. Bônus — Painel Completo de Speedup (todos os grids, todas as abordagens)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
fig.suptitle('Speedup por Tamanho de Grid (todos os algoritmos paralelos)', fontsize=13, fontweight='bold')

for ax, (grid, glabel) in zip(axes, zip(GRID_SIZES, GRID_LABELS)):
    for ab in ['Bidirecional A*', 'HDA*', 'PBNF']:
        sub = agg[(agg['grid_size'] == grid) & (agg['abordagem'] == ab)].sort_values('obst_pct')
        ax.plot(
            OBSTACLE_LABELS, sub['speedup'],
            label=ab, color=COLORS[ab],
            marker=MARKERS[ab], linewidth=2.2, markersize=7
        )
    ax.axhline(1, linestyle='--', color='gray', linewidth=1.2)
    ax.set_title(f'Grid {glabel}')
    ax.set_xlabel('Percentual de Obstáculos')
    if ax == axes[0]:
        ax.set_ylabel('Speedup')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('bonus_speedup_todos_grids.png', bbox_inches='tight')
plt.show()

## 15. Bônus — Boxplot de Variabilidade por Abordagem — Grid 5K×5K
> Mostra a distribuição das 20 execuções independentes

In [ ]:
import matplotlib.patches as mpatches

raw5k = raw[raw['grid_size'] == '5000x5000']

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
fig.suptitle('Variabilidade das Execuções (20 amostras) — Grid 5K×5K', fontsize=13, fontweight='bold')

for ax, obst in zip(axes, OBSTACLE_VALUES):
    data  = [raw5k[(raw5k['abordagem'] == ab) & (raw5k['obst_pct'] == obst)]['tempo_ms'].dropna().values
             for ab in ABORDAGENS]
    bp = ax.boxplot(data, patch_artist=True, notch=False, widths=0.5,
                    medianprops=dict(color='white', linewidth=2))
    for patch, ab in zip(bp['boxes'], ABORDAGENS):
        patch.set_facecolor(COLORS[ab])
        patch.set_alpha(0.85)
    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.set_xticks(range(1, 5))
    ax.set_xticklabels([a.replace(' ', '\n') for a in ABORDAGENS], fontsize=8)
    ax.set_title(f'Obstáculos: {obst}%')
    ax.set_ylabel('Tempo (ms, log)' if obst == 0 else '')

patches = [mpatches.Patch(color=COLORS[ab], label=ab) for ab in ABORDAGENS]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig('bonus_boxplot_variabilidade_5k.png', bbox_inches='tight')
plt.show()

NameError: name 'raw' is not defined